# 08 · Comité CEO / PMO — Economic Intelligence
Notebook ejecutivo para invocar el sistema como una agencia interna **viva**: evidencia → forecast → decisión → experimento → resultado.

La salida debe responder en minutos: **qué cambió, por qué importa, qué pasará, qué decisión requiere, qué riesgo tiene y cómo aprenderemos**.

In [ ]:
from pathlib import Path
import sys, pandas as pd, numpy as np, matplotlib.pyplot as plt
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src'/'replica_cygnus').exists())
sys.path.insert(0, str(ROOT/'src'))
from replica_cygnus.economic_intelligence import install_feature_mart, load_monthly_panel, build_supervised_panel, build_committee_brief
from replica_cygnus.economic_intelligence.features import select_model_matrix
from replica_cygnus.economic_intelligence.models import backtest_regressors, fit_champion_model
install_feature_mart(); panel = load_monthly_panel(); sup = build_supervised_panel(panel, horizons=(3,6))


## Forecast rápido de 6 meses para PMO
El champion **no se fija manualmente**: se selecciona por MAE en holdout temporal reciente. El comité puede cambiar `HORIZON=3/6` sin reescribir el notebook. Si las variables macro todavía están vacías, el feature contract las excluye automáticamente del entrenamiento.

In [ ]:
HORIZON=6
target=f'target_mov_neto_next_{HORIZON}m'
X,y,cols=select_model_matrix(sup,target,include_macro=True,include_reference=False)
meta=sup.loc[X.index,['periodo_mes','codigo_proyecto','proyecto']]
scores,_,pred_holdout=backtest_regressors(X,y,meta,nonnegative_target=False,test_months=6)
champion_name=scores.iloc[0]['name']
model=fit_champion_model(X,y,champion_name)
print(f'Champion PMO: {champion_name}')
display(scores)
latest=sup.sort_values('periodo_mes').groupby('codigo_proyecto',as_index=False).tail(1).copy()
pred=np.clip(model.predict(latest[cols].replace([np.inf,-np.inf],np.nan)),0,None)
fc=latest[['codigo_proyecto','proyecto','periodo_mes','saldo_final_observado']].copy()
fc[f'forecast_demanda_neta_{HORIZON}m']=pred
fc['forecast_demanda_neta_3m']=pred*(3/HORIZON)
fc['forecast_demanda_neta_6m']=pred*(6/HORIZON)
rate=pred/HORIZON
fc['meses_stockout_p50']=latest['saldo_final_observado'].to_numpy()/np.where(rate>0,rate,np.nan)
fc['mes_stockout_p50']=fc.apply(lambda r:(pd.Timestamp(r['periodo_mes']).to_period('M')+int(np.ceil(r['meses_stockout_p50']))).to_timestamp() if pd.notna(r['meses_stockout_p50']) else pd.NaT,axis=1)
brief=build_committee_brief(panel,fc)
brief

## One-page operating review
La tabla siguiente es el producto que debería llegar a un CEO/gerencia: stock, absorción, tendencia, meses de stock, forecast, stockout y banderas de acción. El nombre y métricas del champion mostradas arriba forman parte del gate de gobernanza del comité.

In [ ]:
cols_show=['proyecto','stock_actual_observado','stock_total_actual_ref','absorcion_neta_mes','absorcion_ma6','tendencia_absorcion','meses_stock_a_ritmo_ma3','forecast_demanda_neta_3m','forecast_demanda_neta_6m','mes_stockout_p50','prioridad_pmo','banderas']
brief[cols_show]

In [ ]:
plot=brief.sort_values('meses_stock_a_ritmo_ma3',ascending=False)
fig,ax=plt.subplots(figsize=(11,5))
ax.bar(plot['proyecto'],plot['meses_stock_a_ritmo_ma3'])
ax.axhline(12,linestyle='--',label='12 meses')
ax.set_title(f'PMO · meses de stock al ritmo neto MA3 · champion {champion_name}');ax.set_ylabel('Meses');ax.tick_params(axis='x',rotation=45);ax.legend();plt.show()

## Squads / corrientes del producto
| Corriente | Pregunta | Output | Cadencia |
|---|---|---|---|
| Data Foundation | ¿puedo confiar? | contrato, coverage, DQ | diaria/mensual |
| Microeconomía | ¿qué pasa dentro del proyecto? | oferta, demanda, mix, pricing | semanal |
| Macroeconomía | ¿qué mueve el mercado? | ciclo, tasas, TC, stock agregado | mensual |
| Forecasting ML | ¿qué pasará? | demanda N meses, stockout, minutas | mensual |
| Revenue/Pricing | ¿qué palanca mover? | escenarios y guardrails | quincenal |
| Experimentación | ¿qué causó qué? | A/B, rollout, uplift | por experimento |
| PMO / Decision Intelligence | ¿qué decidimos y aprendimos? | comité, acciones, outcomes | semanal/mensual |

## Script de comité
1. **HECHO:** qué cambió y calidad del dato.
2. **MECANISMO:** demanda, conversión, stock, pricing o macro.
3. **FORECAST:** próximos 3/6 meses + mes de stockout + error holdout del champion.
4. **DECISIÓN:** una acción con responsable y guardrail.
5. **EXPERIMENTO:** qué evidencia distinguirá hipótesis.
6. **OUTCOME:** qué métrica debe moverse y cuándo revisamos.

El notebook no sustituye al decisor: hace explícito el puente entre dato, incertidumbre y decisión.